# MTSamples → Health Assistant Tool-Calling Dataset

Produces a **3-column HuggingFace-style dataset** (`messages` | `tools` | `text`) where each row is a multi-turn function-calling conversation between a patient and a health assistant that uses tools like `getSymptomInfo`, `findMedication`, `checkDrugInteraction`, `bookAppointment`, `getLabTestInfo`, and `findNearbyClinic`.

| Step | Who | Input → Output |
|------|-----|----------------|
| **1** | LLM (Qwen3-32B) | MT clinical case → **patient question** (natural language) |
| **2** | Python | Patient question + specialty → **select relevant health tools** |
| **3** | Python | Execute tools deterministically → **function calls + results** (ground truth) |
| **4** | LLM (Qwen3-32B) | Patient request + trajectory → **`messages`** list (multi-turn FC format) |
| **5** | Python | Validate → save TSV: **messages** \| **tools** \| **text** (`<start_of_turn>` format) |

> Colab Pro **GPU** + `HF_TOKEN` in Secrets. Qwen3-32B loaded via `llama-cpp-python` (A100, 4-bit GGUF).



## 0. Install dependencies


In [ ]:
!pip install -q datasets pandas tqdm huggingface_hub pyarrow
!pip uninstall -y llama-cpp-python
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --no-cache-dir
print("\u2705 Dependencies installed")


## 1. GPU sanity check


In [ ]:
import subprocess, textwrap

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError("\u274c No GPU detected. Change runtime to GPU (A100) in Colab.")

for line in result.stdout.strip().split("\n"):
    name, total, free = [x.strip() for x in line.split(",")]
    print(f"GPU : {name}")
    print(f"VRAM: {total} total  |  {free} free")

print("\n\u2705 GPU ready")


## 2. Imports & configuration


In [ ]:
import os
import json
import re
import time
import textwrap
import csv
import random
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login
from tqdm.auto import tqdm

# HuggingFace auth
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        os.environ["HF_TOKEN"] = hf_token
    except Exception:
        pass
if hf_token:
    login(token=hf_token)
    print("\u2705 HF_TOKEN set")
else:
    print("\u26a0\ufe0f  HF_TOKEN not set \u2014 public datasets will still work")

# Model
MODEL_REPO = "unsloth/Qwen3-32B-GGUF"
GGUF_FILE  = "Qwen3-32B-UD-Q4_K_XL.gguf"

# Dataset
NUM_SAMPLES             = 200   # set None for full dataset
QUESTION_STYLE_SEED     = 42
TRANSCRIPTION_MAX_CHARS = 1500
LLM_MAX_TOKENS          = 200
MESSAGE_MAX_TOKENS      = 4096

# Output
OUT_DIR        = Path("output")
OUT_DIR.mkdir(exist_ok=True)
CSV_DELIM      = "\t"
TOOLCALL_CSV     = OUT_DIR / "health_toolcall_sft.csv"
TOOLCALL_JSONL   = OUT_DIR / "health_toolcall_sft.jsonl"
TOOLCALL_PARQUET = OUT_DIR / "health_toolcall_sft.parquet"
TOOLCALL_HF_DIR  = OUT_DIR / "health_toolcall_sft_hf"
QA_CSV           = OUT_DIR / "health_questions.csv"

print("\u2705 Config loaded")
print(f"   model   : {MODEL_REPO}")
print(f"   samples : {NUM_SAMPLES or 'ALL'}")
print(f"   output  : {TOOLCALL_CSV}")


## 3. Load Qwen3-32B GGUF

4-bit GGUF (~20 GB) via `llama-cpp-python` — fits on a Colab Pro A100 with full GPU offload.



In [ ]:
from llama_cpp import Llama, llama_supports_gpu_offload

if not llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python has no GPU offload \u2014 inference will be very slow. "
        "Fix: Runtime \u2192 Change runtime type \u2192 GPU \u2192 Restart and re-run install cell."
    )

print(f"Loading {MODEL_REPO} / {GGUF_FILE} \u2026")

llm_model = Llama.from_pretrained(
    repo_id=MODEL_REPO,
    filename=GGUF_FILE,
    token=hf_token,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_batch=512,
    verbose=False,
)

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
if smi.returncode == 0:
    used_mb, total_mb = [int(x.strip()) for x in smi.stdout.strip().split(",")]
    print(f"\u2705 Qwen3-32B ready  |  VRAM {used_mb/1024:.1f} / {total_mb/1024:.1f} GB")
    if used_mb < 2000:
        print("\u26a0\ufe0f  VRAM usage low \u2014 model may be on CPU. Re-run install cell.")
else:
    print("\u2705 Qwen3-32B ready")


## 4. Load the MTSamples dataset


In [ ]:
print("Loading harishnair04/mtsamples \u2026")
raw_ds = load_dataset("harishnair04/mtsamples", split="train", token=hf_token)
print(f"\u2705 Loaded {len(raw_ds):,} rows  |  columns: {raw_ds.column_names}")


## 5. Data exploration


In [ ]:
df_raw = raw_ds.to_pandas()
df_raw.head(3)


In [ ]:
print("=== Specialty distribution (top 15) ===")
print(df_raw["medical_specialty"].value_counts().head(15).to_string())
print(f"\n  Total unique specialties: {df_raw['medical_specialty'].nunique()}")


In [ ]:
print("=== Null counts ===")
key_cols = ["description", "medical_specialty", "transcription"]
print(df_raw[key_cols].isnull().sum())
df_raw["transcription_len"] = df_raw["transcription"].str.len().fillna(0).astype(int)
print("\n=== Transcription length ===")
print(df_raw["transcription_len"].describe().to_string())


## 6. Pre-process: filter & clean


In [ ]:
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.dropna(subset=["description", "medical_specialty", "transcription"])
    df = df[df["transcription"].str.len() >= 50]
    df["description"]       = df["description"].str.strip()
    df["medical_specialty"] = df["medical_specialty"].str.strip()
    df["transcription"]     = df["transcription"].str.strip()
    print(f"Cleaned: {before:,} \u2192 {len(df):,} rows  (dropped {before - len(df):,})")
    return df.reset_index(drop=True)


df_clean = clean_df(df_raw.copy())

if NUM_SAMPLES is not None:
    df_sample = df_clean.sample(n=min(NUM_SAMPLES, len(df_clean)), random_state=42)
else:
    df_sample = df_clean.copy()

print(f"Working with {len(df_sample):,} samples")


## 7. Health assistant tool definitions

Six tools a general health assistant would realistically call. Python executes them deterministically (no LLM) to produce ground-truth trajectories.



In [ ]:
HEALTH_TOOLS: list[dict] = [
    {
        "type": "function",
        "function": {
            "name": "getSymptomInfo",
            "description": (
                "Return a structured overview of a medical symptom or condition: "
                "common causes, red-flag signs, typical triage level, and whether "
                "immediate care is needed."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "symptom": {
                        "type": "string",
                        "description": "The symptom or condition to look up (e.g. 'chest pain', 'rash')."
                    },
                    "patient_age_group": {
                        "type": "string",
                        "enum": ["child", "adult", "elderly"],
                        "description": "Broad age group of the patient."
                    },
                },
                "required": ["symptom"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "findMedication",
            "description": (
                "Look up a medication by name or therapeutic class. Returns generic name, "
                "drug class, common uses, typical dosage form, and key warnings."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "medication_name": {
                        "type": "string",
                        "description": "Brand or generic name of the medication."
                    },
                    "include_alternatives": {
                        "type": "boolean",
                        "description": "Whether to include common therapeutic alternatives."
                    },
                },
                "required": ["medication_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "checkDrugInteraction",
            "description": (
                "Check for clinically significant interactions between two or more drugs. "
                "Returns severity (mild / moderate / severe), mechanism, and management advice."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "drugs": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of drug names to check against each other."
                    },
                },
                "required": ["drugs"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "bookAppointment",
            "description": (
                "Schedule a medical appointment for a patient. Returns a booking confirmation "
                "with appointment ID, date-time slot, provider name, and preparation instructions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "specialty": {
                        "type": "string",
                        "description": "Medical specialty required (e.g. 'cardiology', 'dermatology')."
                    },
                    "urgency": {
                        "type": "string",
                        "enum": ["routine", "urgent", "emergency"],
                        "description": "How quickly the appointment is needed."
                    },
                    "patient_name": {
                        "type": "string",
                        "description": "Full name of the patient."
                    },
                },
                "required": ["specialty", "urgency"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "getLabTestInfo",
            "description": (
                "Retrieve information about a laboratory or diagnostic test: what it measures, "
                "normal reference ranges, preparation required, and turnaround time."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "test_name": {
                        "type": "string",
                        "description": "Name of the lab test (e.g. 'complete blood count', 'HbA1c')."
                    },
                    "include_prep": {
                        "type": "boolean",
                        "description": "Whether to include patient preparation instructions."
                    },
                },
                "required": ["test_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "findNearbyClinic",
            "description": (
                "Find clinics or hospitals near the patient that offer a given specialty or service. "
                "Returns a ranked list with name, distance, accepted insurance, and next availability."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "specialty": {
                        "type": "string",
                        "description": "Specialty or service needed (e.g. 'orthopedics', 'urgent care')."
                    },
                    "location": {
                        "type": "string",
                        "description": "City or ZIP code of the patient."
                    },
                    "max_distance_km": {
                        "type": "number",
                        "description": "Maximum search radius in kilometres."
                    },
                },
                "required": ["specialty", "location"],
            },
        },
    },
]

print(f"\u2705 {len(HEALTH_TOOLS)} health tools defined:")
for t in HEALTH_TOOLS:
    print(f"   \u2022 {t['function']['name']}")


## 8. Specialty → tool pipeline mapping & deterministic executors


In [ ]:
# Maps MT specialty to (primary_tool, secondary_tool | None, specialty_label)
SPECIALTY_TOOL_MAP: dict[str, tuple] = {
    "Allergy / Immunology":       ("getSymptomInfo",    "bookAppointment",     "immunology"),
    "Cardiovascular / Pulmonary": ("getSymptomInfo",    "bookAppointment",     "cardiology"),
    "Dentistry":                  ("getSymptomInfo",    "findNearbyClinic",    "dentistry"),
    "Dermatology":                ("getSymptomInfo",    "findMedication",      "dermatology"),
    "Emergency Room Reports":     ("getSymptomInfo",    "findNearbyClinic",    "emergency medicine"),
    "ENT - Otolaryngology":       ("getSymptomInfo",    "bookAppointment",     "ENT"),
    "Gastroenterology":           ("getSymptomInfo",    "getLabTestInfo",      "gastroenterology"),
    "General Medicine":           ("getSymptomInfo",    "bookAppointment",     "general practice"),
    "Hematology - Oncology":      ("getLabTestInfo",    "bookAppointment",     "hematology"),
    "Neurology":                  ("getSymptomInfo",    "bookAppointment",     "neurology"),
    "Obstetrics / Gynecology":    ("getSymptomInfo",    "bookAppointment",     "obstetrics"),
    "Ophthalmology":              ("getSymptomInfo",    "findNearbyClinic",    "ophthalmology"),
    "Orthopedic":                 ("getSymptomInfo",    "findNearbyClinic",    "orthopedics"),
    "Pain Management":            ("getSymptomInfo",    "findMedication",      "pain management"),
    "Pathology":                  ("getLabTestInfo",    "bookAppointment",     "pathology"),
    "Pediatrics - Neonatal":      ("getSymptomInfo",    "bookAppointment",     "pediatrics"),
    "Physical Medicine - Rehab":  ("getSymptomInfo",    "findNearbyClinic",    "rehabilitation"),
    "Podiatry":                   ("getSymptomInfo",    "bookAppointment",     "podiatry"),
    "Psychiatry / Psychology":    ("getSymptomInfo",    "bookAppointment",     "psychiatry"),
    "Radiology":                  ("getLabTestInfo",    "bookAppointment",     "radiology"),
    "Rheumatology":               ("getSymptomInfo",    "findMedication",      "rheumatology"),
    "Sleep Medicine":             ("getSymptomInfo",    "bookAppointment",     "sleep medicine"),
    "Speech - Language":          ("getSymptomInfo",    "bookAppointment",     "speech therapy"),
    "Urology":                    ("getSymptomInfo",    "bookAppointment",     "urology"),
}

DISCARD   = {"Autopsy", "Neurosurgery", "Surgery", "Bariatrics"}
CRISIS_RE = re.compile(r"\b(suicid\w+|self.harm|kill myself|want to die)\b", re.I)
_URGENCY_MAP = {
    "emergency medicine": "emergency",
    "cardiology":         "urgent",
    "psychiatry":         "urgent",
}


# ── Deterministic tool result generators ─────────────────────────────────
def _tool_getSymptomInfo(symptom: str, patient_age_group: str = "adult") -> dict:
    return {
        "symptom": symptom,
        "patient_age_group": patient_age_group,
        "common_causes": [
            f"Primary cause associated with {symptom}",
            "Secondary systemic cause",
            "Lifestyle or environmental trigger",
        ],
        "red_flag_signs": [
            "Sudden severe onset",
            "Associated with loss of consciousness",
            "Worsening despite rest",
        ],
        "triage_level": "urgent" if any(w in symptom.lower() for w in ["chest", "breath", "blood", "severe"]) else "routine",
        "seek_immediate_care": any(w in symptom.lower() for w in ["chest", "breath", "bleed"]),
        "disclaimer": "This is informational only. Always consult a qualified healthcare professional.",
    }


def _tool_findMedication(medication_name: str, include_alternatives: bool = False) -> dict:
    result = {
        "medication_name": medication_name,
        "generic_name": medication_name.lower().split()[0],
        "drug_class": "Consult pharmacist for drug class details",
        "common_uses": [f"Treatment of conditions related to {medication_name}"],
        "dosage_forms": ["oral tablet", "oral capsule"],
        "key_warnings": ["Do not stop without consulting your doctor", "May interact with other medications"],
    }
    if include_alternatives:
        result["alternatives"] = [f"Alternative-A for {medication_name}", f"Alternative-B for {medication_name}"]
    return result


def _tool_checkDrugInteraction(drugs: list) -> dict:
    return {
        "drugs_checked": drugs,
        "interactions_found": len(drugs) > 1,
        "severity": "moderate" if len(drugs) > 1 else "none",
        "mechanism": "Pharmacokinetic interaction \u2014 consult prescriber" if len(drugs) > 1 else "No interaction identified",
        "management": "Monitor closely; inform your doctor of all medications you take.",
        "disclaimer": "Always verify drug interactions with a licensed pharmacist or physician.",
    }


def _tool_bookAppointment(specialty: str, urgency: str, patient_name: str = "Patient") -> dict:
    import hashlib, datetime
    appt_id = "APT-" + hashlib.md5(f"{specialty}{urgency}".encode()).hexdigest()[:6].upper()
    delta   = {"emergency": 0, "urgent": 2, "routine": 7}.get(urgency, 7)
    slot    = (datetime.datetime.now() + datetime.timedelta(days=delta)).strftime("%Y-%m-%d 09:00")
    return {
        "appointment_id": appt_id,
        "patient_name": patient_name,
        "specialty": specialty,
        "urgency": urgency,
        "scheduled_datetime": slot,
        "provider": f"Dr. Smith ({specialty.title()} Department)",
        "location": "Main Medical Center, Building A",
        "preparation": "Bring photo ID, insurance card, and list of current medications.",
        "confirmation": f"Appointment {appt_id} confirmed for {slot}.",
    }


def _tool_getLabTestInfo(test_name: str, include_prep: bool = True) -> dict:
    result = {
        "test_name": test_name,
        "measures": f"Quantifies levels and markers related to {test_name}",
        "normal_range": "Varies by age and sex \u2014 reference ranges provided in lab report",
        "turnaround_time": "24\u201372 hours",
        "specimen_type": "Blood draw (venipuncture) or as indicated",
    }
    if include_prep:
        result["preparation"] = "Fasting for 8\u201312 hours may be required. Avoid strenuous exercise beforehand."
    return result


def _tool_findNearbyClinic(specialty: str, location: str, max_distance_km: float = 10.0) -> dict:
    return {
        "specialty": specialty,
        "location": location,
        "results": [
            {
                "name": f"City {specialty.title()} Center",
                "distance_km": round(max_distance_km * 0.3, 1),
                "accepted_insurance": ["Blue Cross", "Aetna", "United Health"],
                "next_availability": "Tomorrow 10:00 AM",
                "phone": "+1-555-0100",
            },
            {
                "name": f"Regional Medical \u2014 {specialty.title()} Unit",
                "distance_km": round(max_distance_km * 0.7, 1),
                "accepted_insurance": ["Cigna", "Medicare", "Medicaid"],
                "next_availability": "In 3 days",
                "phone": "+1-555-0200",
            },
        ],
        "total_found": 2,
    }


TOOL_EXECUTORS = {
    "getSymptomInfo":       lambda args: _tool_getSymptomInfo(**args),
    "findMedication":       lambda args: _tool_findMedication(**args),
    "checkDrugInteraction": lambda args: _tool_checkDrugInteraction(**args),
    "bookAppointment":      lambda args: _tool_bookAppointment(**args),
    "getLabTestInfo":       lambda args: _tool_getLabTestInfo(**args),
    "findNearbyClinic":     lambda args: _tool_findNearbyClinic(**args),
}

print("\u2705 Tool executors ready")


## 9. Step 1 — Generate patient question (LLM)

From each MT sample produce a **realistic patient message** in varied voice.



In [ ]:
QUESTION_SYSTEM_PROMPT = textwrap.dedent("""\
    You write realistic patient-style messages for a medical Q&A dataset.
    Each message should sound like something a real person would type to a nurse,
    health chatbot, or search engine \u2014 NOT like a medical exam question.
    Rules:
    - Use everyday language; describe symptoms or worries drawn from the case.
    - Never name a medical specialty or department in the message.
    - Do NOT start with "Which specialty", "Which department", "What type of specialist",
      or "What kind of doctor" \u2014 vary the openings.
    - 1\u20132 short sentences only.
    - Output ONLY the patient's message \u2014 no reasoning, labels, markdown, or quotes.
""")

QUESTION_STYLES = [
    {"id": "first_person",    "instruction": "First-person patient describing their own symptoms.",
     "example": "I've had a dull headache behind my eyes for three days \u2014 should I get this checked out?"},
    {"id": "family_member",   "instruction": "Someone asking about a parent, spouse, or relative.",
     "example": "My mom says her back has been stiff every morning and it's hard to bend \u2014 who should she see?"},
    {"id": "child_caregiver", "instruction": "Parent or caregiver asking about a child's symptoms.",
     "example": "My 6-year-old keeps pulling at his ear and crying at night \u2014 is that urgent?"},
    {"id": "casual_worry",    "instruction": "Casual everyday wording; a bit vague but grounded in the case.",
     "example": "This cough won't go away and I'm wiped out \u2014 could it be more than a cold?"},
    {"id": "after_visit",     "instruction": "Still having problems after ER/clinic; confused what to do next.",
     "example": "They sent me home from the ER but I'm still short of breath when I walk \u2014 what do I do now?"},
    {"id": "symptom_story",   "instruction": "Brief symptom story that ends in a natural question.",
     "example": "Food felt stuck when I swallowed yesterday and it still hurts \u2014 is that normal?"},
    {"id": "friend_asking",   "instruction": "Asking on behalf of a friend in third person.",
     "example": "My friend fainted once last week and felt dizzy after \u2014 what kind of appointment does she need?"},
    {"id": "seeking_help",    "instruction": "Direct plea for help without clinical jargon.",
     "example": "I'm really scared about these palpitations when I lie down \u2014 where should I start?"},
]
_style_rng = random.Random(QUESTION_STYLE_SEED)

THINK_BLOCK   = re.compile(r"<think>.*?</think>",  re.DOTALL | re.IGNORECASE)
PREAMBLE_RE   = re.compile(
    r"^(?:question|here(?:'s| is)(?: the)? question|output|response|answer|patient message)\s*:?\s*",
    re.IGNORECASE,
)
GENERIC_WHICH = re.compile(
    r"^(which|what)\s+(specialty|department|type of (doctor|specialist|physician)|kind of doctor)\b",
    re.IGNORECASE,
)


def pick_style(idx=None):
    return _style_rng.choice(QUESTION_STYLES) if idx is None else QUESTION_STYLES[idx % len(QUESTION_STYLES)]


def build_q_prompt(description: str, transcription: str, style: dict) -> str:
    trunc = transcription[:TRANSCRIPTION_MAX_CHARS]
    if len(transcription) > TRANSCRIPTION_MAX_CHARS:
        trunc += "... [truncated]"
    return (
        f"DESCRIPTION:\n{description}\n\n"
        f"TRANSCRIPTION:\n{trunc}\n\n"
        f"VOICE: {style['instruction']}\n"
        f"Tone example (do not copy verbatim): {style['example']}\n\n"
        "Write 1\u20132 sentences as that person would actually speak or type.\n"
        "Ground it in the case facts above. Do not name any specialty."
    )


def extract_question(text: str):
    if not text or not text.strip():
        return None
    text = THINK_BLOCK.sub("", text)
    text = re.sub(r"</?think>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```(?:json|text)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text)
    text = PREAMBLE_RE.sub("", text).strip().strip("\"'`")
    text = re.sub(r"\s+", " ", text)
    parts = re.split(r"(?<=[.!?])\s+", text)
    parts = [p.strip() for p in parts if p.strip()]
    if not parts:
        return None
    q = parts[0]
    if len(parts) > 1 and len(q) < 80 and not q.endswith("?"):
        q = f"{q} {parts[1]}"
    q = q.strip().strip("\"'`")
    if not re.search(r"[.!?]$", q):
        q = q.rstrip(".") + "?"
    return q if len(q) >= 20 else None


def llm(user_prompt: str, max_tokens: int = LLM_MAX_TOKENS, temp: float = 0.55) -> str:
    out = llm_model.create_chat_completion(
        messages=[
            {"role": "system", "content": QUESTION_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=max_tokens,
        temperature=temp,
        top_p=0.9,
    )
    return out["choices"][0]["message"].get("content") or ""


def generate_patient_question(description: str, transcription: str, style_idx=None):
    style = pick_style(style_idx)
    raw   = llm(build_q_prompt(description, transcription, style))
    q     = extract_question(raw)
    if q and GENERIC_WHICH.match(q.strip()):
        alt   = pick_style((style_idx or 0) + 3)
        q     = extract_question(llm(build_q_prompt(description, transcription, alt), temp=0.65))
        style = alt
    return q, style["id"]


print("\u2705 Patient question helpers ready")
print(f"   question styles  : {len(QUESTION_STYLES)}")
print(f"   max new tokens   : {LLM_MAX_TOKENS}")


## 10. Steps 2–3 — Select and execute health tools (Python)

Each specialty maps to 1–2 tools. Python executes them deterministically to produce **ground-truth tool calls + results**.



In [ ]:
def extract_health_context(row: dict, row_idx: int = 0):
    specialty = str(row.get("medical_specialty", "")).strip()
    if specialty in DISCARD or specialty not in SPECIALTY_TOOL_MAP:
        return None
    desc = str(row.get("description", "")).strip()
    if len(desc) < 20 or CRISIS_RE.search(desc):
        return None
    primary_tool, secondary_tool, specialty_label = SPECIALTY_TOOL_MAP[specialty]
    condition = str(row.get("sample_name") or desc).strip()[:120]
    return {
        "medical_specialty": specialty,
        "specialty_label":   specialty_label,
        "condition":         condition,
        "primary_tool":      primary_tool,
        "secondary_tool":    secondary_tool,
        "source_row_id":     row_idx,
    }


def build_tool_args(tool_name: str, ctx: dict, patient_query: str) -> dict:
    spec    = ctx["specialty_label"]
    cond    = ctx["condition"]
    urgency = _URGENCY_MAP.get(spec, "routine")
    if tool_name == "getSymptomInfo":
        return {"symptom": cond, "patient_age_group": "adult"}
    if tool_name == "findMedication":
        return {"medication_name": cond.split()[0], "include_alternatives": True}
    if tool_name == "checkDrugInteraction":
        return {"drugs": [cond.split()[0], "aspirin"]}
    if tool_name == "bookAppointment":
        return {"specialty": spec, "urgency": urgency, "patient_name": "Patient"}
    if tool_name == "getLabTestInfo":
        return {"test_name": cond, "include_prep": True}
    if tool_name == "findNearbyClinic":
        return {"specialty": spec, "location": "your area", "max_distance_km": 15.0}
    return {}


def run_health_tool_pipeline(ctx: dict, patient_query: str) -> dict:
    tool_steps = []
    step_num   = 1
    for tool_name in filter(None, [ctx["primary_tool"], ctx["secondary_tool"]]):
        args   = build_tool_args(tool_name, ctx, patient_query)
        result = TOOL_EXECUTORS[tool_name](args)
        tool_steps.append({
            "step":   step_num,
            "name":   tool_name,
            "args":   args,
            "result": result,
        })
        step_num += 1
    return {
        "tool_steps":     tool_steps,
        "primary_tool":   ctx["primary_tool"],
        "secondary_tool": ctx["secondary_tool"],
    }


print("\u2705 Health tool pipeline ready")


## 11. Step 4 — Generate message sequence (LLM)

Output format matches the HuggingFace function-calling SFT style shown in the dataset viewer:

- **system** — `"You are a helpful assistant with access to the following functions. Use them if required -"` + tools JSON in `content`
- **user / assistant** — alternating turns only
- **assistant tool calls** — JSON array in `content`: `[{"name": "...", "arguments": {...}}]`
- **tool results** — returned as **user** messages



In [ ]:
FC_SYSTEM_PREFIX = (
    "You are a helpful assistant with access to the following functions. "
    "Use them if required -\n"
)

MESSAGE_SEQUENCE_SYSTEM_PROMPT = textwrap.dedent("""\
    You build function-calling training transcripts for a health assistant dataset.
    Given a patient request and GROUND-TRUTH tool trajectory, output JSON: {"messages": [...]}.

    REQUIRED FORMAT (roles: system, user, assistant ONLY \u2014 no "tool" role):
    1) system  \u2014 content = FC_SYSTEM_PREFIX + tools JSON array
    2) user    \u2014 patient question verbatim
    3) assistant \u2014 call the first tool: content = JSON array only, e.g.
       [{"name": "getSymptomInfo", "arguments": {"symptom": "chest pain", "patient_age_group": "adult"}}]
    4) user    \u2014 first tool result JSON
    5) assistant \u2014 (if secondary tool) second tool call JSON array
    6) user    \u2014 second tool result JSON (omit turns 5\u20136 if no secondary tool)
    7) assistant \u2014 final helpful plain-text response summarising findings

    Tool-call assistant messages must be ONLY a JSON array string \u2014 no markdown fences.
    Never diagnose, never name controlled substances, never prescribe.
    Return ONLY valid JSON.
""").strip()


def build_system_message(tools: list) -> dict:
    return {
        "role":    "system",
        "content": FC_SYSTEM_PREFIX + json.dumps(tools, ensure_ascii=False, indent=2),
    }


def is_tool_call_content(content) -> bool:
    if not content or not isinstance(content, str):
        return False
    text = content.strip()
    if not text.startswith("["):
        return False
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return False
    return isinstance(data, list) and bool(data) and isinstance(data[0], dict) and "name" in data[0]


def assistant_tool_call(name: str, arguments: dict) -> dict:
    payload = [{"name": name, "arguments": arguments}]
    return {"role": "assistant", "content": json.dumps(payload, ensure_ascii=False, indent=2)}


def user_tool_result(result: dict) -> dict:
    return {"role": "user", "content": json.dumps(result, ensure_ascii=False)}


def assemble_messages_fallback(patient_query: str, ctx: dict, traj: dict) -> list:
    """Deterministic FC-format transcript \u2014 always used as ground truth."""
    messages = [
        build_system_message(HEALTH_TOOLS),
        {"role": "user", "content": patient_query},
    ]
    for step in traj["tool_steps"]:
        messages.append(assistant_tool_call(step["name"], step["args"]))
        messages.append(user_tool_result(step["result"]))
    # Final assistant summary
    last_result = traj["tool_steps"][-1]["result"]
    if "confirmation" in last_result:
        summary = (
            f"Good news \u2014 {last_result['confirmation']} "
            "Please remember to bring your ID and insurance card. "
            "Let me know if you need anything else."
        )
    elif "results" in last_result:
        top = last_result["results"][0] if last_result["results"] else {}
        summary = (
            f"I found {last_result.get('total_found', 0)} clinic(s) near you. "
            f"The closest is {top.get('name', 'a nearby clinic')} "
            f"({top.get('distance_km', '?')} km away), "
            f"available as early as {top.get('next_availability', 'soon')}. "
            "Would you like me to book an appointment?"
        )
    elif "triage_level" in last_result:
        summary = (
            f"Based on the information, your concern has a triage level of "
            f"'{last_result['triage_level']}'. "
            + ("Please seek immediate care." if last_result.get("seek_immediate_care") else
               "A routine appointment should be sufficient.")
            + " Always consult a qualified healthcare professional for a proper diagnosis."
        )
    else:
        summary = (
            "Here is what I found based on your question. "
            "Please consult a qualified healthcare professional for personalised advice."
        )
    messages.append({"role": "assistant", "content": summary})
    return messages


def build_trajectory_prompt(patient_query: str, ctx: dict, traj: dict) -> str:
    steps_txt = []
    for s in traj["tool_steps"]:
        steps_txt.append(
            f"Step {s['step']}: {s['name']}\n"
            f"  args  : {json.dumps(s['args'])}\n"
            f"  result: {json.dumps(s['result'])[:600]}"
        )
    example_call = json.dumps([{"name": "getSymptomInfo", "arguments": {"symptom": "chest pain"}}], indent=2)
    return (
        f"PATIENT QUESTION (verbatim for turn 2):\n{patient_query}\n\n"
        f"SPECIALTY: {ctx['specialty_label']}  |  CONDITION: {ctx['condition']}\n\n"
        "GROUND-TRUTH TOOL TRAJECTORY:\n" + "\n".join(steps_txt) + "\n\n"
        f"TOOL CALL FORMAT EXAMPLE:\n{example_call}\n\n"
        "Generate the messages JSON for health assistant function-calling fine-tuning."
    )


def parse_messages_json(raw: str):
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m:
            return None
        try:
            data = json.loads(m.group())
        except json.JSONDecodeError:
            return None
    msgs = data.get("messages") if isinstance(data, dict) else None
    if not isinstance(msgs, list) or len(msgs) < 4:
        return None
    return msgs


def llm_sequence(user_prompt: str, max_tokens: int = MESSAGE_MAX_TOKENS) -> str:
    out = llm_model.create_chat_completion(
        messages=[
            {"role": "system", "content": MESSAGE_SEQUENCE_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.35,
        top_p=0.9,
    )
    return out["choices"][0]["message"].get("content") or ""


def validate_messages(messages: list, patient_query: str) -> bool:
    if not messages or messages[0].get("role") != "system":
        return False
    if FC_SYSTEM_PREFIX[:30] not in (messages[0].get("content") or ""):
        return False
    allowed = {"system", "user", "assistant"}
    if {m.get("role") for m in messages} - allowed:
        return False
    user_c = [m["content"] for m in messages if m.get("role") == "user" and m.get("content")]
    if not user_c or patient_query[:30] not in user_c[0]:
        return False
    return True


MODEL_ACK = "Okay, got it!"


def render_text_column(messages: list) -> str:
    """Full Gemma <start_of_turn> training string."""
    system_content = None
    dialogue = []
    for m in messages:
        if m.get("role") == "system":
            system_content = (m.get("content") or "").rstrip()
        elif m.get("role") in ("user", "assistant"):
            dialogue.append(m)
    if not system_content:
        system_content = FC_SYSTEM_PREFIX + json.dumps(HEALTH_TOOLS, ensure_ascii=False, indent=2)
    turns = [
        f"<start_of_turn>user\n{system_content}\n<end_of_turn>",
        f"<start_of_turn>model\n{MODEL_ACK}\n<end_of_turn>",
    ]
    for m in dialogue:
        tag = "user" if m["role"] == "user" else "model"
        turns.append(f"<start_of_turn>{tag}\n{(m.get('content') or '').rstrip()}\n<end_of_turn>")
    return "\n".join(turns)


def generate_messages_sequence(patient_query: str, ctx: dict, traj: dict):
    base   = assemble_messages_fallback(patient_query, ctx, traj)
    prompt = build_trajectory_prompt(patient_query, ctx, traj)
    raw    = llm_sequence(prompt)
    llm_msgs = parse_messages_json(raw)
    if llm_msgs and validate_messages(llm_msgs, patient_query):
        llm_prose = [
            m["content"] for m in llm_msgs
            if m.get("role") == "assistant" and not is_tool_call_content(m.get("content"))
        ]
        out, pi = [], 0
        for m in base:
            item = dict(m)
            if item.get("role") == "assistant" and not is_tool_call_content(item.get("content")):
                if pi < len(llm_prose) and llm_prose[pi]:
                    item["content"] = llm_prose[pi]
                pi += 1
            out.append(item)
        return out, "llm+ground_truth"
    return base, "ground_truth"


print("\u2705 Message sequence generator ready")


## 12. Smoke test (run before the full loop)


In [ ]:
_smoke_row = df_sample.iloc[0].to_dict()
_smoke_ctx = extract_health_context(_smoke_row, 0)
assert _smoke_ctx, "Smoke: context extraction failed"
_smoke_traj = run_health_tool_pipeline(_smoke_ctx, "I have been having chest pain.")
assert _smoke_traj["tool_steps"], "Smoke: tool pipeline produced no steps"
_smoke_msgs = assemble_messages_fallback("I have been having chest pain.", _smoke_ctx, _smoke_traj)
print(f"Smoke FC format:")
print(f"  turns : {len(_smoke_msgs)} | roles: {[m['role'] for m in _smoke_msgs]}")
print(f"  system starts: {_smoke_msgs[0]['content'][:80]}...")
_smoke_text = render_text_column(_smoke_msgs)
print(f"  text column (first 500 chars):\n{_smoke_text[:500]}...")


## 13. Build all records

**Step 1** patient question \u2192 **Steps 2\u20133** tool pipeline \u2192 **Step 4** messages \u2192 collect.



In [ ]:
def build_health_record(row: dict, patient_query: str, q_style: str, row_idx: int = 0):
    ctx  = extract_health_context(row, row_idx)
    if ctx is None:
        return None
    traj = run_health_tool_pipeline(ctx, patient_query)
    messages, msg_source = generate_messages_sequence(patient_query, ctx, traj)
    text = render_text_column(messages)
    return {
        "messages": messages,
        "tools":    HEALTH_TOOLS,
        "text":     text,
        "metadata": {
            "patient_query":     patient_query,
            "question_style":    q_style,
            "medical_specialty": ctx["medical_specialty"],
            "specialty_label":   ctx["specialty_label"],
            "condition":         ctx["condition"],
            "primary_tool":      ctx["primary_tool"],
            "secondary_tool":    ctx["secondary_tool"] or "",
            "message_source":    msg_source,
            "message_turns":     len(messages),
            "text_chars":        len(text),
            "source_row_id":     ctx["source_row_id"],
        },
    }


health_records = []
skipped        = 0
rows_list      = df_sample.to_dict("records")
t_start        = time.time()

for i, row in enumerate(tqdm(rows_list, desc="Building health tool-call records")):
    try:
        patient_query, q_style = generate_patient_question(
            row["description"], row["transcription"], style_idx=i,
        )
    except Exception as e:
        print(f"\n\u26a0\ufe0f  Row {i} skipped (LLM): {e}")
        patient_query, q_style = None, None

    if not patient_query:
        skipped += 1
        continue

    rec = build_health_record(row, patient_query, q_style, row_idx=i)
    if rec is None:
        skipped += 1
        continue

    health_records.append(rec)

elapsed = time.time() - t_start
print(f"\n\u2705 Built {len(health_records):,} records  |  skipped {skipped}")
print(f"   Total time : {elapsed/60:.1f} min")

if health_records:
    print("\nTool mix (primary):")
    print(pd.Series([r["metadata"]["primary_tool"] for r in health_records]).value_counts().to_string())


## 14. Inspect results


In [ ]:
assert health_records, "No records \u2014 check smoke test above"
df_meta = pd.DataFrame([r["metadata"] for r in health_records])
df_meta.head(10)


In [ ]:
print("=== Primary tool distribution ===")
print(df_meta["primary_tool"].value_counts().to_string())
print()
print("=== Medical specialty (top 10) ===")
print(df_meta["medical_specialty"].value_counts().head(10).to_string())


In [ ]:
print("=== Sample records ===\n")
for rec in random.sample(health_records, min(2, len(health_records))):
    m = rec["metadata"]
    print(f"[{m['question_style']}]  specialty={m['medical_specialty']}")
    print(f"  tools   : {m['primary_tool']} \u2192 {m['secondary_tool'] or 'none'}")
    print(f"  turns   : {m['message_turns']} | text chars: {m['text_chars']}")
    print(f"  PATIENT : {m['patient_query']}")
    print(f"  FINAL   : {rec['messages'][-1]['content'][:200]}")
    tool_calls = [x for x in rec["messages"] if is_tool_call_content(x.get("content"))]
    if tool_calls:
        print(f"  TOOL #1 : {tool_calls[0]['content'][:150]}...")
    print("-" * 70)


## 15. Save outputs

3-column table matching the HuggingFace dataset viewer layout:

| Column | Type | Description |
|--------|------|-------------|
| `messages` | list | Full conversation JSON (all turns + tool args + results) |
| `tools` | list | Health tool function definitions |
| `text` | string | Gemma `<start_of_turn>` training string |



In [ ]:
def records_to_dataset_df(records: list) -> pd.DataFrame:
    return pd.DataFrame({
        "messages": [r["messages"] for r in records],
        "tools":    [r["tools"]    for r in records],
        "text":     [r["text"]     for r in records],
    })


def save_csv(path, records: list) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=CSV_DELIM)
        w.writerow(["messages", "tools", "text"])
        for rec in records:
            w.writerow([
                json.dumps(rec["messages"], ensure_ascii=False),
                json.dumps(rec["tools"],    ensure_ascii=False),
                rec["text"],
            ])


save_csv(TOOLCALL_CSV, health_records)
print(f"\u2705 CSV      \u2192 {TOOLCALL_CSV}")

df_dataset = records_to_dataset_df(health_records)
print(f"\u2705 Dataset table: {len(df_dataset)} rows \u00d7 {list(df_dataset.columns)}")
print(f"   avg turns      : {df_dataset['messages'].map(len).mean():.1f}")
print(f"   avg text chars : {df_dataset['text'].str.len().mean():.0f}")

with open(TOOLCALL_JSONL, "w", encoding="utf-8") as f:
    for rec in health_records:
        f.write(json.dumps({
            "messages": rec["messages"],
            "tools":    rec["tools"],
            "text":     rec["text"],
            "metadata": rec["metadata"],
        }, ensure_ascii=False) + "\n")
print(f"\u2705 JSONL    \u2192 {TOOLCALL_JSONL}")

try:
    df_dataset.to_parquet(TOOLCALL_PARQUET, index=False)
    print(f"\u2705 Parquet  \u2192 {TOOLCALL_PARQUET}")
except Exception as e:
    print(f"\u26a0\ufe0f  Parquet skipped ({e}) \u2014 run: pip install pyarrow")

try:
    from datasets import Dataset
    Dataset.from_pandas(df_dataset).save_to_disk(str(TOOLCALL_HF_DIR))
    print(f"\u2705 HF dataset \u2192 {TOOLCALL_HF_DIR}")
except Exception as e:
    print(f"\u26a0\ufe0f  HF dataset save skipped ({e})")

pd.DataFrame([r["metadata"] for r in health_records]).to_csv(QA_CSV, index=False)
print(f"\u2705 Metadata \u2192 {QA_CSV}")


In [ ]:
# Preview the 3-column table \u2014 mirrors the HuggingFace dataset viewer
df_dataset.head(3)


## 16. Quality checks


In [ ]:
df_meta["q_len"] = df_meta["patient_query"].str.len()
print("Patient question length (chars):")
print(df_meta["q_len"].describe().round(1).to_string())


In [ ]:
def specialty_leaked(question: str, specialty: str) -> bool:
    words = [w.lower() for w in specialty.split() if len(w) > 3]
    return any(w in question.lower() for w in words)


df_meta["leaked"] = df_meta.apply(
    lambda r: specialty_leaked(r["patient_query"], r["medical_specialty"]), axis=1
)
leak_rate = df_meta["leaked"].mean()
print(f"Specialty leak rate : {leak_rate:.1%}  ({df_meta['leaked'].sum()} / {len(df_meta)})")

dupes      = df_meta["patient_query"].duplicated().sum()
generic    = df_meta["patient_query"].str.match(GENERIC_WHICH).sum()
has_tool   = sum(
    1 for r in health_records
    if any(is_tool_call_content(m.get("content")) for m in r["messages"])
)
llm_src    = sum(1 for r in health_records if "llm" in r["metadata"].get("message_source", ""))

print(f"Duplicate questions : {dupes} / {len(df_meta)}")
print(f"Generic Which\u2026      : {generic} / {len(df_meta)}")
print(f"Has tool call turn  : {has_tool} / {len(health_records)}")
print(f"LLM prose polish    : {llm_src} / {len(health_records)}")


In [ ]:
print("\n=== FINAL DATASET SUMMARY ===")
print(f"  Model           : {MODEL_REPO}")
print(f"  Records         : {len(health_records):,}")
print(f"  Tools           : {len(HEALTH_TOOLS)}")
print(f"  Specialty leak  : {leak_rate:.1%}")
print(f"  Avg question    : {df_meta['q_len'].mean():.0f} chars")
print(f"\nOutputs:")
for p in [TOOLCALL_CSV, TOOLCALL_JSONL, QA_CSV]:
    print(f"  {p}")


---
## Optional: tool distribution chart


In [ ]:
import matplotlib.pyplot as plt

counts = df_meta["primary_tool"].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
counts.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Count")
ax.set_title("Health tool calls \u2014 dataset distribution")
plt.tight_layout()
plt.savefig(OUT_DIR / "tool_distribution.png", dpi=120)
plt.show()


---
## Optional: download outputs (Colab only)


In [ ]:
try:
    from google.colab import files
    for p in [TOOLCALL_CSV, TOOLCALL_PARQUET, TOOLCALL_JSONL, QA_CSV]:
        files.download(str(p))
except ImportError:
    print("Not in Colab \u2014 files are in:", OUT_DIR.resolve())
